* Import necessary libraries

In [33]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

* Load data

In [18]:
data = pd.read_csv("../../datasets/processed/diabetes_cleaned.csv")
print(data.info())
print(data.duplicated().sum())
print(data.gender.unique())

<class 'pandas.DataFrame'>
RangeIndex: 96128 entries, 0 to 96127
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               96128 non-null  str    
 1   age                  96128 non-null  float64
 2   hypertension         96128 non-null  int64  
 3   heart_disease        96128 non-null  int64  
 4   smoking_history      96128 non-null  str    
 5   bmi                  96128 non-null  float64
 6   HbA1c_level          96128 non-null  float64
 7   blood_glucose_level  96128 non-null  int64  
 8   diabetes             96128 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.6 MB
None
0
<StringArray>
['Female', 'Male']
Length: 2, dtype: str


* Separate Feature and Target variables

In [19]:
X = data.drop("diabetes", axis=1)
y = data["diabetes"]

In [20]:
X.dtypes

gender                     str
age                    float64
hypertension             int64
heart_disease            int64
smoking_history            str
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
dtype: object

*Two variables, 'Gender' and 'Smoking history' to be encoded*

* Split data into train and test

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [22]:
# check on split data

print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(67289, 8)
(28839, 8)
diabetes
0    0.911769
1    0.088231
Name: proportion, dtype: float64
diabetes
0    0.911751
1    0.088249
Name: proportion, dtype: float64


* Separate feature types

In [23]:
categorical_features = ["gender", "smoking_history"]
numerical_features = ["age", "hypertension", "heart_disease", "bmi", "HbA1c_level", "blood_glucose_level"]

## Logistic Regression Model
------
(Baseline Model)

* Preprocessing

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

* Logistic Regression pipeline

In [25]:
log_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

* Training and prediction

In [26]:
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

* Accuracy check

In [27]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9575227989874823


*A higher accuracy in this case does not necessarily indicate that the model performs well, due to the substantial class imbalance between diabetic and non-diabetic observations. Since the majority class consists of non-diabetic observations, the model can achieve relatively high accuracy by predicting more observations as non-diabetic, while potentially failing to correctly identify diabetic cases.*

In [34]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification report
print(classification_report(y_test, y_pred))

[[26006   288]
 [  937  1608]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     26294
           1       0.85      0.63      0.72      2545

    accuracy                           0.96     28839
   macro avg       0.91      0.81      0.85     28839
weighted avg       0.95      0.96      0.95     28839



*The baseline Logistic Regression model distinguishes non-diabetic cases very well, but its ability to detect diabetic cases is substantially weaker, with 63% recall.*

* Handling class imbalance

(Balanced Logistic Regression model)

In [46]:
log_reg_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight={0: 1, 1: 3},
        max_iter=1000
    ))
])

In [47]:
log_reg_balanced.fit(X_train, y_train)

y_pred = log_reg_balanced.predict(X_test)

In [48]:
accuracy = accuracy_score(y_test, y_pred)
print("\n Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print("\n Confusion matrix:", cm)

print(classification_report(y_test, y_pred))



 Accuracy: 0.9414334755019245

 Confusion matrix: [[25252  1042]
 [  647  1898]]
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     26294
           1       0.65      0.75      0.69      2545

    accuracy                           0.94     28839
   macro avg       0.81      0.85      0.83     28839
weighted avg       0.95      0.94      0.94     28839



*class_weight="balanced", and manually tuned class weight substantially increased the model's ability to detect class 1, but it also produced many more false positives.*

## Random Forest Classifier
------
(Baseline Model)

* Preprocessing

In [50]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

* Random Forest pipeline

In [51]:
random_forest = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

* Train and Predict

In [52]:
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)

In [53]:
accuracy = accuracy_score(y_test, y_pred)
print("\n Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print("\n Confusion matrix:", cm)

print(classification_report(y_test, y_pred))



 Accuracy: 0.9690696626096605

 Confusion matrix: [[26197    97]
 [  795  1750]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     26294
           1       0.95      0.69      0.80      2545

    accuracy                           0.97     28839
   macro avg       0.96      0.84      0.89     28839
weighted avg       0.97      0.97      0.97     28839

